# A3 end-to-end check: Basira's real scorer on real models

Runs `basira_ai.acoustic.Listener` (the code in this branch) on synthetic Google-TTS clips and, optionally, your voice
notes. **Synthetic TTS = adult synthetic voice; voice notes = adult team member. No child speech.**

Runtime → Change runtime type → T4 GPU → Connect → Runtime → Run all. Copy the block printed by the last cell.

In [ ]:
%%capture
!rm -rf Kuwait-Dyslexia && git clone -q --depth 1 --branch feat/a3-listen https://github.com/DXstr/Kuwait-Dyslexia.git
!pip -q install -e "Kuwait-Dyslexia/ai[model]" gTTS

In [ ]:
import os, shutil, sys
sys.path.insert(0, "Kuwait-Dyslexia/ai")
os.environ["BASIRA_CONTRACTS_DIR"] = os.path.abspath("Kuwait-Dyslexia/contracts")
from gtts import gTTS
from basira_ai.acoustic import Listener, load_audio
listener = Listener()
print("models loaded")
shutil.rmtree("clips", ignore_errors=True); os.makedirs("clips")

In [ ]:
CASES = [  # (id, expected text, spoken text)
  ("qalam_msa", "قَلَم", "قَلَم"),
  ("thaa_ok", "ثَمِيك", "ثَمِيك"), ("thaa_as_taa", "ثَمِيك", "تَمِيك"),
  ("dhaal_ok", "ذَرُوم", "ذَرُوم"), ("dhaal_as_daal", "ذَرُوم", "دَرُوم"),
  ("khaa_ok", "خَلُوم", "خَلُوم"), ("khaa_as_hhaa", "خَلُوم", "حَلُوم"),
  ("saad_ok", "صَمُوك", "صَمُوك"), ("saad_as_siin", "صَمُوك", "سَمُوك"),
  ("jiim_ok", "دَجَاج", "دَجَاج"), ("jiim_as_y", "دَجَاج", "دَيَاي"),
  ("sentence", "ذَهَبَ الْوَلَدُ إِلَى الْمَدْرَسَةِ", "ذَهَبَ الْوَلَدُ إِلَى الْمَدْرَسَةِ"),
]
clips = []
for cid, expected, spoken in CASES:
    path = f"clips/{cid}.mp3"; gTTS(spoken, lang="ar").save(path)
    clips.append((cid, expected, spoken, path, "SYNTHETIC gTTS-ar"))
print(len(clips), "synthetic clips")

Optional: upload your voice notes again (`qalam_q`, `qalam_g`, `qalam_hamza`, `dajaj_y`). Click **Cancel upload** to skip.

In [ ]:
REC = {"qalam_q": "قَلَم", "qalam_g": "قَلَم", "qalam_hamza": "قَلَم", "dajaj_y": "دَجَاج"}
try:
    from google.colab import files
    up = files.upload()
except Exception as e:
    up = {}
for name, data in up.items():
    key = next((k for k in REC if name.lower().startswith(k)), None)
    if key:
        path = f"clips/rec_{name}"; open(path, "wb").write(data)
        clips.append(("rec_" + key, REC[key], "(recording)", path, "REAL adult voice note (Egyptian speaker)"))
print(len(clips), "clips total")

In [ ]:
# ===== COPY EVERYTHING PRINTED BELOW AND PASTE IT TO CLAUDE =====
print("A3-LISTEN-CHECK v1")
for cid, expected, spoken, path, source in clips:
    print("---", cid, "| expected", expected, "| spoken", spoken, "|", source)
    try:
        ev = listener.listen(load_audio(path), expected)
    except Exception as e:
        print("  ERROR", type(e).__name__, e); continue
    for e in ev:
        if e.unit.kind != "consonant": continue
        flag = "" if e.winner_ipa == e.unit.ipa else "  <-- heard " + e.winner_ipa + (f" ({e.winner_letter})" if e.winner_letter else "")
        print(f"  {e.unit.letter:6s} exp {e.unit.ipa:3s} score {e.score:.2f} by {e.decided_by:6s}{' DEGRADED' if e.degraded else ''}{flag}")